# Fundamentals 2 · Neurons and Activation Functions

A single neuron computes a weighted sum $z = x \cdot w + b$ and passes it through an activation function.
This notebook works through that computation by hand, then in NumPy, then for a whole batch at once, and
finally designs neurons by hand that compute logic gates — the starting point for Experiment 1.

## A single neuron, by hand

Take a neuron with weights $w = (3, -1.5, 2)$, bias $b = -2$ and the input $x = (1, 2, 3)$.

**Weighted sum:**

*$z = (w_1*x_1) + (w_2*x_2) + (w_3*x_3) + b$*  
*$z = (3*1) + (-1.5*2) + (2*3) + (-2)$*  
*$z = 3 - 3 + 6 - 2$*  
*$z = 4$*

**Step** activation:

*$\sigma(a) = step(a)$*  
*$\hat{y} = \sigma(z)$*  
*$\hat{y} = \sigma(4)$*  
*$\hat{y} = 1$*

**Sigmoid** activation:

*$\sigma(a) = sigmoid(a) = \frac{1}{1 + e^{-x}}$*  
*$\hat{y} = \sigma(z)$*  
*$\hat{y} = \sigma(4)$*  
*$\hat{y} = \frac{1}{1 + e^{-4}}$*  
*$\hat{y} = 0.982$*

**ReLU** activation:

*$\sigma = ReLU(a) = max(0, a)$*  
*$\hat{y} = \sigma(z)$*  
*$\hat{y} = \sigma(4)$*  
*$\hat{y} = max(0, 4)$*  
*$\hat{y} = 4$*  

## The same neuron in NumPy

The same computation in code, with one function per activation, checked against the values above.

In [1]:
import numpy as np

weights = np.array([3, -1.5, 2])
bias = -2

test_inputs = np.array([1, 2, 3])

Weighted sum $z$:

In [2]:
def compute_z(inputs, w, b):
    return np.dot(inputs, w) + b

z = compute_z(test_inputs, weights, bias)

print(z)

4.0


Step:

In [3]:
def step(a):
    return (a >= 0).astype(int)

print(step(z))

1


Sigmoid:

In [4]:
def sigmoid(a):
    return 1 / (1 + np.exp(-a))

print(sigmoid(z))

0.9820137900379085


ReLU:

In [5]:
def relu(a):
    return np.maximum(0, a)

print(relu(z))

4.0


## A batch of inputs

Stacking inputs as the rows of a matrix $X$ processes a whole batch at once: $X w + b$ gives one $z$ per
row. The second record here is $(-2, -4, -6)$; only ReLU is shown.

In [6]:
batch_inputs = np.array([[1, 2, 3], [-2, -4, -6]])
weights = np.array([3, -1.5, 2])

transposed_weights = weights.T

batch_z = compute_z(batch_inputs, transposed_weights, bias)
print(f"z for the batch: {batch_z}")

# ReLU activation function
print(f"ReLU for the batch: {relu(batch_z)}")

z for the batch: [  4. -14.]
ReLU for the batch: [4. 0.]


## Logic gates from a single neuron

Computers are built from logic gates that combine bits. With true = 1 and false = 0, the AND gate outputs 1
only when both inputs are 1. A single neuron with a step activation can compute it: with $w = (1, 1)$ and
$b = -1.5$, $z$ is positive only for $(1, 1)$. The bias is chosen so that no $z$ is exactly 0, where the step
function jumps.

| $x_1$ | $x_2$ | $x_1$ AND $x_2$ | $z = x*w^T + b$ | $y = step(z)$ |
| --- | --- | --- | --- | --- |
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | -0.5 | 0 |
| 1 | 0 | 0 | -0.5 | 0 |
| 1 | 1 | 1 | 2 | 1 |

The code below confirms the table.

In [7]:
and_inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
weights_and = np.array([1, 1])
transposed_weights_and = weights_and.T
bias = -1.5

and_z = compute_z(and_inputs, transposed_weights_and, bias)
print(f"z for AND gate: {and_z}")

and_step = step(and_z)
print(f"step for AND gate: {and_step}")

z for AND gate: [-1.5 -0.5 -0.5  0.5]
step for AND gate: [0 0 0 1]


### Designing an OR gate

OR outputs 1 when at least one input is 1. Keeping $w = (1, 1)$, any bias between $-1$ and $0$ works;
$b = -0.5$ gives:

| $x_1$ | $x_2$ | $x_1$ OR $x_2$ | $z = x*w^T + b$ | $y = step(z)$ |
| --- | --- | --- | --- | --- |
| 0 | 0 | 0 | -0.5 | 0 |
| 0 | 1 | 1 | 0.5 | 1 |
| 1 | 0 | 1 | 0.5 | 1 |
| 1 | 1 | 1 | 1.5 | 1 |

$w = [1, 1]$  
$b = -0.5$

## Reflection: choosing weights by hand

I didn't use trial and error. Instead, I looked at the pattern from the AND gate. For the OR gate, $z$ has to
be at least 0 whenever one input is 1 and negative otherwise, because of how the step activation works.
Keeping the AND gate's weights ($w_1 = 1$, $w_2 = 1$), the weighted sums before the bias are $[0, 1, 1, 2]$. That
leaves only the bias, which has to push the all-zero case below 0 while keeping every other case above 0, so
any value in $(-1, 0)$ works. With $b = -0.5$, $z = [-0.5, 0.5, 0.5, 1.5]$.

Finding weights by hand only worked because the OR gate has two inputs and four cases, so the pattern was
easy to reason out. It does not scale to a model like VGG16 from Fundamentals 1, with 138 million weights and
biases: there is no truth table to inspect and trial and error is hopeless. Those weights are *learned* from
data — gradient descent (Fundamentals 4) nudges every weight downhill on a cost function, and
backpropagation (Fundamentals 5–7) computes all of those gradients efficiently.